In [7]:
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)
import torch

# === 1. Load Dataset ===
dataset = load_dataset(
    "json",
    data_files=r"D:\LingoMalay\Models_Transcribe\Dataset\Kedah\#Main(clean)\##Transcripts2.jsonl",
    split="train"
)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# === 2. Load Processor (tokenizer + feature extractor) ===
# r"D:\LingoMalay\Models_Transcribe\Model\Kelantan\whisper-kelantanv1v2\checkpoint-50"
processor = WhisperProcessor.from_pretrained("mesolitica/malaysian-whisper-small-v3")
processor.tokenizer.add_special_tokens({'additional_special_tokens': ['<|kedah|>', '<|kelantan|>']})

tokenizer = processor.tokenizer

# === 3. Load Model and Resize Token Embeddings ===
# D:\LingoMalay\Models_Transcribe\Model\Kelantan\whisper-kelantan
model = WhisperForConditionalGeneration.from_pretrained(r"mesolitica/malaysian-whisper-small-v3")
model.resize_token_embeddings(len(tokenizer))

# === 4. Preprocessing Function ===
def preprocess(batch):
    audio = batch["audio"]

    # Audio features
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]

    # Prompt-based target text
    prompt = "<|startoftranscript|><|ms|><|kelantan|><|transcribe|>"
    full_text = prompt + batch["text"].strip() + " <|endoftext|>"
    # full_text = batch["text"].strip()
    batch["labels"] = tokenizer(full_text).input_ids
    return batch

dataset = dataset.map(preprocess, remove_columns=dataset.column_names)
dataset.set_format(type="torch")

# === 5. Data Collator ===
def data_collator(batch):
    input_features = torch.stack([torch.tensor(item["input_features"]) for item in batch])

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        processor.tokenizer.pad_token = tokenizer.eos_token

    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
    labels = torch.nn.utils.rnn.pad_sequence(
        labels, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    labels[labels == tokenizer.pad_token_id] = -100  # Mask pad tokens from loss

    return {"input_features": input_features, "labels": labels}

# Freeze the encoder layers
# for param in model.model.encoder.parameters():
#     param.requires_grad = False

# === 6. Training Arguments ===
training_args = Seq2SeqTrainingArguments(
    output_dir="./Model/Kedah/whisper-kelantanv2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-5, #-5 early rate, lower when fine-tuning
    warmup_steps=100,
    max_steps=1000,
    save_steps=200,
    logging_steps=200,
    fp16=True,  # Set to False if GPU doesn't support fp16
    # save_total_limit=2,
    report_to="none"
)

# === 7. Trainer Setup ===
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=processor.tokenizer,
    data_collator=data_collator
)

# === 8. Start Training ===
trainer.train()

Generating train split: 71 examples [00:00, 5109.13 examples/s]
c:\Users\aqils\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Map: 100%|██████████| 71/71 [00:01<00:00, 52.42 examples/s]
c:\Users\aqils\AppData\Local\Programs\Python\Python310\lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batch

{'loss': 2.3147, 'grad_norm': 16.856836318969727, 'learning_rate': 8.955555555555555e-06, 'epoch': 2.82}


C:\Users\aqils\AppData\Local\Temp\ipykernel_8072\1625237329.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_features = torch.stack([torch.tensor(item["input_features"]) for item in batch])
C:\Users\aqils\AppData\Local\Temp\ipykernel_8072\1625237329.py:55: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
 40%|████      | 400/1000 [01:46<03:26,  2.91it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be r

{'loss': 0.1772, 'grad_norm': 11.59327220916748, 'learning_rate': 6.733333333333334e-06, 'epoch': 5.63}


C:\Users\aqils\AppData\Local\Temp\ipykernel_8072\1625237329.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_features = torch.stack([torch.tensor(item["input_features"]) for item in batch])
C:\Users\aqils\AppData\Local\Temp\ipykernel_8072\1625237329.py:55: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
 60%|██████    | 600/1000 [03:04<02:44,  2.44it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be r

...

C:\Users\aqils\AppData\Local\Temp\ipykernel_8072\1625237329.py:49: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_features = torch.stack([torch.tensor(item["input_features"]) for item in batch])
C:\Users\aqils\AppData\Local\Temp\ipykernel_8072\1625237329.py:55: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]
100%|██████████| 1000/1000 [05:52<00:00,  2.46it/s]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be 

{'loss': 0.0005, 'grad_norm': 0.007843081839382648, 'learning_rate': 6.666666666666668e-08, 'epoch': 14.08}


100%|██████████| 1000/1000 [05:57<00:00,  2.80it/s]

{'train_runtime': 357.0601, 'train_samples_per_second': 2.801, 'train_steps_per_second': 2.801, 'train_loss': 0.5019818745031953, 'epoch': 14.08}


TrainOutput(global_step=1000, training_loss=0.5019818745031953, metrics={'train_runtime': 357.0601, 'train_samples_per_second': 2.801, 'train_steps_per_second': 2.801, 'train_loss': 0.5019818745031953, 'epoch': 14.08})

# LingoMalay 🎙️

LingoMalay is a specialized mobile application developed using Flutter designed to record, upload, and transcribe Malay speech, with a specific focus on regional dialects such as **Kedah** and **Kelantan**. 

Built for research purposes, this project explores dialect-sensitive Automatic Speech Recognition (ASR) by leveraging fine-tuned Whisper models and integrating with Mesolitica's cutting-edge APIs. The backend architecture relies on Firebase for secure user authentication, structured data storage, and real-time state management.

---

## 🚀 Features

- **Dialect-Specific Transcription:** Tailored handling of Kedah and Kelantan speech variations.
- **Real-Time Inference:** High-accuracy Malay audio-to-text processing powered by fine-tuned OpenAI Whisper models and Mesolitica APIs.
- **Audio Management:** Native-level audio recording, playback, and local file handling within a Flutter interface.
- **Robust Cloud Infrastructure:** Fully integrated with Firebase Auth for user management and Firebase Storage/Firestore for audio assets and transcription metadata.

---

## 📁 Repository Structure

Based on the project's architecture, the repository is organized into the following key modules:

```text
├── App/                             # Flutter Mobile Application
│   ├── android/, ios/, web/         # Native platform configurations
│   ├── assets/                      # Global assets (icons, placeholder audio, etc.)
│   └── lib/                      
│        ├── models/                 # Data structures 
│        ├── provider/               # State management 
│        ├── routes/                 # App navigation mapping
│        ├── screens/                # UI Views 
│        ├── themes/                 # App styling and color palettes
│        ├── utils/                  # Helper functions
│        ├── widgets/                # Reusable UI components
│        ├── firebase_options.dart               
│        └── main.dart            
└── Models_Transcribe/   
    ├── API/                        # Core backend API services and endpointsand server deployment configurations sent to Google Cloud (GDC)
    ├── Model/                      # Fine-tuned Whisper model checkpoints
    ├── Dataset/                    # Specialized audio datasets
    └── NoteBook/                   # Jupyter Notebooks used for model training, evaluation, and inference testing
